## CPSC 4970 - Jonathan Braun - M6 Notebook 2: SOM + MLP

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [14]:
penguins = pd.read_csv("penguins.csv", na_values=["?", "_"])

print("Original shape:", penguins.shape)
print("\nMissing values before dropping:")
print(penguins.isna().sum())

penguins = penguins.dropna().copy()

print("\nCleaned shape:", penguins.shape)
print("\nMissing values after dropping:")
print(penguins.isna().sum())

penguins.head()

Original shape: (344, 7)

Missing values before dropping:
species               0
island                0
culmen_length_mm      2
culmen_depth_mm       2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

Cleaned shape: (333, 7)

Missing values after dropping:
species              0
island               0
culmen_length_mm     0
culmen_depth_mm      0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64


,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE


In [15]:
numeric_features = [
    "culmen_length_mm",
    "culmen_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

categorical_features = [
    "island",
    "sex"
]

X_numeric = penguins[numeric_features]
X_categorical = penguins[categorical_features]
y = penguins["species"]

print("Numerical features:")
display(X_numeric.head())

print("Categorical features:")
display(X_categorical.head())

print("Target counts:")
print(y.value_counts())

Numerical features:


,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0
4,36.7,19.3,193.0,3450.0
5,39.3,20.6,190.0,3650.0


Categorical features:


,island,sex
0,Torgersen,MALE
1,Torgersen,FEMALE
2,Torgersen,FEMALE
4,Torgersen,FEMALE
5,Torgersen,MALE


Target counts:
species
Adelie       146
Gentoo       119
Chinstrap     68
Name: count, dtype: int64


## Train/test split

In [16]:
X_train_numeric, X_test_numeric, X_train_categorical, X_test_categorical, y_train, y_test = train_test_split(
    X_numeric,
    X_categorical,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training numerical shape:", X_train_numeric.shape)
print("Testing numerical shape:", X_test_numeric.shape)
print("Training categorical shape:", X_train_categorical.shape)
print("Testing categorical shape:", X_test_categorical.shape)

Training numerical shape: (249, 4)
Testing numerical shape: (84, 4)
Training categorical shape: (249, 2)
Testing categorical shape: (84, 2)


In [17]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_numeric)
X_test_scaled = scaler.transform(X_test_numeric)

print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled testing data shape:", X_test_scaled.shape)

Scaled training data shape: (249, 4)
Scaled testing data shape: (84, 4)


## SOM implementation

In [18]:
class SimpleSOM:
    def __init__(self, m, n, dim, learning_rate=0.5, sigma=None, n_iterations=3000, random_state=None):
        self.m = m
        self.n = n
        self.dim = dim
        self.learning_rate = learning_rate
        self.sigma = sigma if sigma is not None else max(m, n) / 2
        self.n_iterations = n_iterations
        self.random_state = random_state
        self.rng = np.random.default_rng(random_state)
        self.locations = np.array([(i, j) for i in range(m) for j in range(n)])
        self.weights = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)

        # Initialize SOM weights using random samples from the training data
        initial_indices = self.rng.choice(len(X), size=self.m * self.n, replace=True)
        self.weights = X[initial_indices].copy()

        for t in range(self.n_iterations):
            x = X[self.rng.integers(len(X))]
            bmu_index = self.winner_index(x)

            progress = t / max(1, self.n_iterations - 1)
            current_learning_rate = self.learning_rate * np.exp(-progress)
            current_sigma = max(self.sigma * np.exp(-progress), 1e-6)

            distances_squared = np.sum(
                (self.locations - self.locations[bmu_index]) ** 2,
                axis=1
            )

            neighborhood = np.exp(
                -distances_squared / (2 * current_sigma * current_sigma)
            )[:, np.newaxis]

            self.weights += current_learning_rate * neighborhood * (x - self.weights)

        return self

    def winner_index(self, x):
        distances = np.linalg.norm(self.weights - x, axis=1)
        return int(np.argmin(distances))

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self.winner_index(x) for x in X])

    def labels_as_coordinates(self, X):
        winner_indices = self.predict(X)
        coordinates = self.locations[winner_indices]
        return np.array([f"{row}_{col}" for row, col in coordinates])

## Train the SOM and create categories

In [19]:
som = SimpleSOM(
    m=3,
    n=3,
    dim=4,
    learning_rate=0.5,
    sigma=1.5,
    n_iterations=3000,
    random_state=7
)

som.fit(X_train_scaled)

train_som_categories = som.labels_as_coordinates(X_train_scaled)
test_som_categories = som.labels_as_coordinates(X_test_scaled)

print("Training SOM categories:")
print(pd.Series(train_som_categories).value_counts().sort_index())

print("\nTesting SOM categories:")
print(pd.Series(test_som_categories).value_counts().sort_index())

Training SOM categories:
0_0    34
0_1    14
0_2    30
1_0    43
1_1    13
1_2    23
2_0    38
2_1    18
2_2    36
Name: count, dtype: int64

Testing SOM categories:
0_0    14
0_1     4
0_2     7
1_0    10
1_1     7
1_2    10
2_0    16
2_1     3
2_2    13
Name: count, dtype: int64


## Build the MLP input data

In [20]:
X_train_mlp = X_train_categorical.copy()
X_test_mlp = X_test_categorical.copy()

X_train_mlp["som_category"] = train_som_categories
X_test_mlp["som_category"] = test_som_categories

print("MLP training features:")
display(X_train_mlp.head())

print("MLP testing features:")
display(X_test_mlp.head())

MLP training features:


,island,sex,som_category
185,Dream,MALE,0_1
39,Dream,MALE,1_0
319,Biscoe,MALE,0_2
89,Dream,FEMALE,1_0
169,Dream,FEMALE,0_0


MLP testing features:


,island,sex,som_category
190,Dream,FEMALE,0_0
289,Biscoe,MALE,0_2
112,Biscoe,FEMALE,2_1
236,Biscoe,FEMALE,2_2
203,Dream,MALE,0_1


## Encode the categorical features and train the MLP

In [21]:
mlp_features = ["island", "sex", "som_category"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), mlp_features)
    ]
)

mlp_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", MLPClassifier(
            hidden_layer_sizes=(20, 10),
            max_iter=1000,
            random_state=42
        ))
    ]
)

mlp_model.fit(X_train_mlp, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[<U9](3,)","['Adelie','Chinstrap','Gentoo']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['island','sex','som_category']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere 

## Accuracy

In [22]:
train_predictions = mlp_model.predict(X_train_mlp)
test_predictions = mlp_model.predict(X_test_mlp)

train_accuracy = accuracy_score(y_train, train_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)

print("Training accuracy:", train_accuracy)
print("Testing accuracy:", test_accuracy)

Training accuracy: 0.9839357429718876
Testing accuracy: 0.9880952380952381


## Confusion matrices

In [23]:
species_labels = sorted(y.unique())

train_cm = pd.DataFrame(
    confusion_matrix(y_train, train_predictions, labels=species_labels),
    index=[f"Actual {label}" for label in species_labels],
    columns=[f"Predicted {label}" for label in species_labels]
)

test_cm = pd.DataFrame(
    confusion_matrix(y_test, test_predictions, labels=species_labels),
    index=[f"Actual {label}" for label in species_labels],
    columns=[f"Predicted {label}" for label in species_labels]
)

print("Training confusion matrix:")
display(train_cm)

print("Testing confusion matrix:")
display(test_cm)

Training confusion matrix:


,Predicted Adelie,Predicted Chinstrap,Predicted Gentoo
Actual Adelie,109,0,0
Actual Chinstrap,4,47,0
Actual Gentoo,0,0,89


Testing confusion matrix:


,Predicted Adelie,Predicted Chinstrap,Predicted Gentoo
Actual Adelie,37,0,0
Actual Chinstrap,1,16,0
Actual Gentoo,0,0,30


## Classification report

In [24]:
print("Classification report:")
print(classification_report(y_test, test_predictions))

Classification report:
              precision    recall  f1-score   support

      Adelie       0.97      1.00      0.99        37
   Chinstrap       1.00      0.94      0.97        17
      Gentoo       1.00      1.00      1.00        30

    accuracy                           0.99        84
   macro avg       0.99      0.98      0.99        84
weighted avg       0.99      0.99      0.99        84



In [25]:
print("Final MLP input columns:")
print(X_train_mlp.columns.tolist())

Final MLP input columns:
['island', 'sex', 'som_category']


## SOM and MLP conclusion

After dropping rows with missing values, the cleaned dataset contained 333 penguins. I split the data into a training set of 249 rows and a testing set of 84 rows. For this part of the assignment, the four numerical features, culmen length, culmen depth, flipper length, and body mass, were used only to train the SOM. These original numerical features were not used directly as input to the MLP.

The numerical features were standardized and then used to train a 3 x 3 SOM. This reduced each penguin’s four-dimensional numerical measurements to a single categorical SOM value. The resulting SOM category was then combined with the two non-numerical features, island and sex. The final MLP input features were island, sex, and SOM category. These three categorical columns were one-hot encoded before being passed into the MLP classifier.

The MLP classifier performed very well. It achieved a training accuracy of approximately 98.39% and a testing accuracy of approximately 98.81%. On the testing set, the model correctly classified all Adelie penguins, all Gentoo penguins, and 16 out of 17 Chinstrap penguins. The only testing error was one Chinstrap penguin classified as Adelie.

These results suggest that the SOM category captured useful information from the numerical measurements, and that combining this category with island and sex produced a strong feature set for the MLP classifier. Overall, the SOM and MLP approach performed very well on this data.

## Citation

Horst AM, Hill AP, Gorman KB (2020). palmerpenguins: Palmer Archipelago (Antarctica) penguin data. R package version 0.1.0. https://allisonhorst.github.io/palmerpenguins/. doi: 10.5281/zenodo.3960218.